# DigiMistCudaTensor — PyTorch MNIST (GPU) MLP with Keras-like knobs

This notebook implements a Dense/MLP classifier for MNIST with parameters inspired by Keras (`kernel_initializer`, `bias_initializer`, `kernel_regularizer`, `bias_regularizer`, `BatchNormalization`, `Dropout`, `EarlyStopping`, etc.).

### Key requirements implemented:
* **Dataset:** `torchvision.datasets.MNIST`
* **Labels:** one-hot encoded to match Keras `CategoricalCrossentropy`
* **Output activation:** Softmax (probabilities)
* **Loss:** Categorical Cross-Entropy implemented as `-(y * log(p)).sum(dim=-1).mean()`
* **Architecture:** 784 → 750 → 500 → 300 → 150 → 75 → 30 → 10
* **Activations:** ReLU or LeakyReLU (tunable)
* **Initializers:** Xavier/Glorot uniform or He normal (tunable)
* **Regularization:** L1 on kernel and bias (tunable)
* **Advanced feature:** LoRA adapters for each Dense layer (tunable `lora_rank`, `lora_alpha`)
* **BatchNorm + Dropout** (default p=0.3, tunable)
* **Early stopping** by accuracy or loss (tunable thresholds)
* **History columns per epoch:** `['Accuracy', 'loss', 'val_Accuracy', 'val_loss', 'epoch_time', 'epoch']`
* **Plots:** accuracy/loss curves + confusion matrix

## GPU utilization notes

To maximize GPU utilization, this notebook uses:
* `pin_memory=True` and `non_blocking=True` transfers
* `torch.backends.cudnn.benchmark=True` (fast convolution heuristics; harmless here but kept as a general speed flag)
* Mixed precision training via `torch.cuda.amp.autocast` + `GradScaler`
* Larger validation batch size derived from training batch size

Quantization is intentionally skipped (`quantization=None`) per your instruction.


In [ ]:
# Imports + Environment

import os
import time
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

import torchvision
from torchvision import transforms

import matplotlib.pyplot as plt
import seaborn as sns

print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type == 'cuda':
    print('cuda:', torch.version.cuda)
    print('gpu:', torch.cuda.get_device_name(0))
    print('capability:', torch.cuda.get_device_capability(0))

# Performance flags
torch.backends.cudnn.benchmark = True

# Reproducibility (still allows cuDNN benchmark; exact determinism not guaranteed)
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)

# Optional: show GPU memory summary helper
def print_cuda_memory(prefix: str = '') -> None:
    if device.type != 'cuda':
        return
    allocated = torch.cuda.memory_allocated() / (1024**2)
    reserved = torch.cuda.memory_reserved() / (1024**2)
    print(f"{prefix} CUDA memory allocated={allocated:.1f}MB reserved={reserved:.1f}MB")

print_cuda_memory('After imports:')


In [ ]:
# ==============================================================================
# 1) Data: MNIST + one-hot labels
# ==============================================================================

def one_hot(labels: torch.Tensor, num_classes: int = 10) -> torch.Tensor:
    # labels: (N,) int64
    return F.one_hot(labels.to(torch.int64), num_classes=num_classes).to(torch.float32)

def build_mnist_dataloaders(
    data_dir: str = './data',
    batch_size: int = 64,
    val_split: float = 0.1,
    num_workers: int = 2,
    pin_memory: bool = True,
    persistent_workers: bool = True,
    prefetch_factor: int = 4,
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """Returns train_loader, val_loader, test_loader.
    
    Notes:
    - We keep labels as integers in the dataset, then one-hot encode inside the training loop.
      This avoids storing one-hot vectors in RAM and matches Keras behavior at loss time.
    - Validation batch size is derived from training batch size (see training function).
    """
    
    tfs = transforms.Compose([
        transforms.ToTensor(), # (1,28,28) float32 in [0,1]
    ])
    
    train_full = torchvision.datasets.MNIST(root=data_dir, train=True, download=True, transform=tfs)
    test_ds = torchvision.datasets.MNIST(root=data_dir, train=False, download=True, transform=tfs)
    
    val_len = int(len(train_full) * val_split)
    train_len = len(train_full) - val_len
    train_ds, val_ds = random_split(train_full, [train_len, val_len], generator=torch.Generator().manual_seed(42))
    
    # Dataloader knobs for throughput
    common = dict(
        num_workers=num_workers,
        pin_memory=(pin_memory and device.type == 'cuda'),
        persistent_workers=(persistent_workers and num_workers > 0),
        prefetch_factor=prefetch_factor if num_workers > 0 else None,
    )
    
    # Remove prefetch_factor if num_workers==0 (PyTorch complains)
    if common['prefetch_factor'] is None:
        common.pop('prefetch_factor')
        
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False, **common)
    # val/test loaders are created in training function to match derived validation batch size
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False, **common)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, drop_last=False, **common)
    
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = build_mnist_dataloaders(batch_size=64)
xb, yb = next(iter(train_loader))
print('x batch:', xb.shape, xb.dtype, xb.min().item(), xb.max().item())
print('y batch:', yb.shape, yb.dtype, yb.min().item(), yb.max().item())


In [ ]:
# ==============================================================================
# 2) Layers: Dense vs Dense+LoRA
# ==============================================================================

class LoRALinear(nn.Module):
    """Linear layer with optional LoRA adapters.
    
    If lora_rank is None or 0, behaves like nn.Linear.
    
    LoRA formulation:
    y = x@W^T + b + scale * (x @ A^T @ B^T)
    where A: (r, in_features), B: (out_features, r)
    scale = lora_alpha / r
    """
    
    def __init__(
        self,
        in_features: int,
        out_features: int,
        bias: bool = True,
        lora_rank: Optional[int] = None,
        lora_alpha: float = 1.0,
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.lora_rank = int(lora_rank) if (lora_rank is not None) else 0
        self.lora_alpha = float(lora_alpha)
        
        self.base = nn.Linear(in_features, out_features, bias=bias)
        
        if self.lora_rank > 0:
            # A: rank x in, B: out x rank
            self.lora_A = nn.Parameter(torch.empty(self.lora_rank, in_features))
            self.lora_B = nn.Parameter(torch.empty(out_features, self.lora_rank))
            self.scaling = self.lora_alpha / self.lora_rank
            # Common LoRA init: A ~ Kaiming/normal, B = zeros so initial delta is 0
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B)
        else:
            self.lora_A = None
            self.lora_B = None
            self.scaling = 0.0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.base(x)
        if self.lora_rank > 0:
            # x: (N, in)
            # x @ A^T -> (N, r)
            # (N, r) @ B^T -> (N, out)
            delta = (x @ self.lora_A.t()) @ self.lora_B.t()
            y = y + self.scaling * delta
        return y

def init_linear_weights(
    linear: nn.Linear,
    kernel_init: str = 'glorot_uniform',
    bias_init: str = 'zeros',
) -> None:
    """Keras-like initializers for Linear weights."""
    k = kernel_init.lower()
    if k in ('glorot_uniform', 'xavier_uniform'):
        nn.init.xavier_uniform_(linear.weight)
    elif k in ('he_normal', 'kaiming_normal'):
        nn.init.kaiming_normal_(linear.weight, nonlinearity='relu')
    else:
        raise ValueError(f"Unsupported kernel_init={kernel_init}. Use 'glorot_uniform' or 'he_normal'.")

    b = bias_init.lower()
    if linear.bias is not None:
        if b == 'zeros':
            nn.init.zeros_(linear.bias)
        else:
            raise ValueError(f"Unsupported bias_init={bias_init}. Use 'zeros'.")

def init_lora_linear(
    layer: LoRALinear,
    kernel_init: str = 'glorot_uniform',
    bias_init: str = 'zeros',
) -> None:
    init_linear_weights(layer.base, kernel_init=kernel_init, bias_init=bias_init)

print('LoRALinear ready')


In [ ]:
# ==============================================================================
# 3) Model: MLP with BN + Dropout
# ==============================================================================

class MNISTMLP(nn.Module):
    def __init__(
        self,
        hidden_sizes: List[int],
        num_classes: int = 10,
        activation: str = 'relu',
        leaky_relu_slope: float = 0.01,
        dropout_p: float = 0.3,
        use_batchnorm: bool = True,
        kernel_init: str = 'glorot_uniform',
        bias_init: str = 'zeros',
        lora_rank: Optional[int] = None,
        lora_alpha: float = 1.0,
        output_softmax: bool = True,
    ):
        super().__init__()
        
        self.output_softmax = bool(output_softmax)
        self.activation_name = activation.lower()
        self.leaky_relu_slope = float(leaky_relu_slope)
        
        sizes = [28 * 28] + list(hidden_sizes) + [num_classes]
        
        layers: List[nn.Module] = []
        for i in range(len(sizes) - 1):
            in_f, out_f = sizes[i], sizes[i + 1]
            is_last = (i == len(sizes) - 2)
            
            dense = LoRALinear(
                in_features=in_f,
                out_features=out_f,
                bias=True,
                lora_rank=lora_rank,
                lora_alpha=lora_alpha,
            )
            init_lora_linear(dense, kernel_init=kernel_init, bias_init=bias_init)
            layers.append(dense)
            if not is_last:
                if use_batchnorm:
                    layers.append(nn.BatchNorm1d(out_f))
                    
                if self.activation_name == 'relu':
                    layers.append(nn.ReLU(inplace=True))
                elif self.activation_name in ('leaky_relu', 'leakyrelu'):
                    layers.append(nn.LeakyReLU(negative_slope=self.leaky_relu_slope, inplace=True))
                else:
                    raise ValueError("activation must be 'relu' or 'leaky_relu'")
                    
                layers.append(nn.Dropout(p=dropout_p))
                
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (N, 1, 28, 28)
        x = x.view(x.size(0), -1)
        logits = self.net(x)
        if self.output_softmax:
            return F.softmax(logits, dim=-1)
        return logits

def build_model(
    kernel_init: str = 'glorot_uniform',
    bias_init: str = 'zeros',
    activation: str = 'relu',
    dropout_p: float = 0.3,
    use_batchnorm: bool = True,
    lora_rank: Optional[int] = None,
    lora_alpha: float = 1.0,
    output_softmax: bool = True,
) -> MNISTMLP:
    hidden = [750, 500, 300, 150, 75, 30]
    model = MNISTMLP(
        hidden_sizes=hidden,
        activation=activation,
        dropout_p=dropout_p,
        use_batchnorm=use_batchnorm,
        kernel_init=kernel_init,
        bias_init=bias_init,
        lora_rank=lora_rank,
        lora_alpha=lora_alpha,
        output_softmax=output_softmax,
    )
    return model

model = build_model(kernel_init='glorot_uniform', activation='relu', lora_rank=None)
print(model)
print('params:', sum(p.numel() for p in model.parameters()))


In [ ]:
# ==============================================================================
# 4) Loss: CategoricalCrossentropy + L1 regs
# ==============================================================================

def categorical_crossentropy_from_probs(
    probs: torch.Tensor,
    y_onehot: torch.Tensor,
    eps: float = 1e-7,
) -> torch.Tensor:
    """Keras-like categorical crossentropy.
    
    probs: (N,C) probabilities (Softmax output)
    y_onehot: (N,C) one-hot labels
    """
    probs = probs.clamp(min=eps, max=1.0)
    return -(y_onehot * probs.log()).sum(dim=-1).mean()

def l1_regularization(
    model: nn.Module,
    kernel_l1: float = 0.0,
    bias_l1: float = 0.0,
) -> torch.Tensor:
    """Adds L1 penalty for weights (kernel) and biases.
    
    We apply L1 to:
    - base Linear weights/biases
    - LoRA A/B matrices (treated as kernel weights)
    """
    k = float(kernel_l1)
    b = float(bias_l1)
    if k == 0.0 and b == 0.0:
        return torch.tensor(0.0, device=device)
        
    reg = torch.tensor(0.0, device=device)
    for m in model.modules():
        if isinstance(m, LoRALinear):
            if k != 0.0:
                reg = reg + k * m.base.weight.abs().sum()
                if m.lora_A is not None:
                    reg = reg + k * m.lora_A.abs().sum()
                if m.lora_B is not None:
                    reg = reg + k * m.lora_B.abs().sum()
            if b != 0.0 and m.base.bias is not None:
                reg = reg + b * m.base.bias.abs().sum()
                
    return reg

print('Loss + L1 regularization ready')


In [ ]:
# ==============================================================================
# 5) Training utilities: metrics + early stop
# ==============================================================================

def accuracy_from_probs(probs: torch.Tensor, y_true_int: torch.Tensor) -> float:
    preds = probs.argmax(dim=-1)
    correct = (preds == y_true_int).sum().item()
    return correct / y_true_int.numel()

@dataclass
class EarlyStopConfig:
    monitor: str = 'accuracy'  # 'accuracy' or 'loss'
    acc_threshold: float = 0.95
    loss_threshold: float = 0.1
    patience: int = 3
    min_delta: float = 0.0

class EarlyStopper:
    def __init__(self, cfg: EarlyStopConfig):
        self.cfg = cfg
        self.best = None
        self.bad_epochs = 0
        
    def should_stop(self, epoch_metrics: Dict[str, float]) -> Tuple[bool, str]:
        m = self.cfg.monitor.lower()
        if m == 'accuracy':
            current = float(epoch_metrics['val_Accuracy'])
            # threshold stop
            if current >= self.cfg.acc_threshold:
                return True, f"EarlyStop: val_Accuracy {current:.4f} >= {self.cfg.acc_threshold:.4f}"
            # patience stop (maximize)
            if self.best is None or current > self.best + self.cfg.min_delta:
                self.best = current
                self.bad_epochs = 0
            else:
                self.bad_epochs += 1
                if self.bad_epochs >= self.cfg.patience:
                    return True, f"EarlyStop: no val_Accuracy improvement for {self.cfg.patience} epochs"
            return False, ""
            
        if m == 'loss':
            current = float(epoch_metrics['val_loss'])
            current = float(epoch_metrics['val_loss'])
            if current <= self.cfg.loss_threshold:
                return True, f"EarlyStop: val_loss {current:.4f} <= {self.cfg.loss_threshold:.4f}"
            # patience stop (minimize)
            if self.best is None or current < self.best - self.cfg.min_delta:
                self.best = current
                self.bad_epochs = 0
            else:
                self.bad_epochs += 1
                if self.bad_epochs >= self.cfg.patience:
                    return True, f"EarlyStop: no val_loss improvement for {self.cfg.patience} epochs"
            return False, ""
            
        raise ValueError("monitor must be 'accuracy' or 'loss'")

print('Early stopping ready')


In [ ]:
# ==============================================================================
# 6) Train/Eval loops (AMP + GPU throughput)
# ==============================================================================

def run_one_epoch_train(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: torch.cuda.amp.GradScaler,
    kernel_l1: float,
    bias_l1: float,
    amp: bool = True,
) -> Tuple[float, float]:
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_n = 0
    
    for xb, yb_int in loader:
        xb = xb.to(device, non_blocking=True)
        yb_int = yb_int.to(device, non_blocking=True)
        yb_oh = one_hot(yb_int, num_classes=10)
        
        optimizer.zero_grad(set_to_none=True)
        
        with torch.cuda.amp.autocast(enabled=(amp and device.type == 'cuda')):
            probs = model(xb)
            loss = categorical_crossentropy_from_probs(probs, yb_oh)
            loss = loss + l1_regularization(model, kernel_l1=kernel_l1, bias_l1=bias_l1)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        bs = yb_int.size(0)
        total_loss += loss.detach().item() * bs
        total_correct += (probs.detach().argmax(dim=-1) == yb_int).sum().item()
        total_n += bs
        
    return total_loss / total_n, total_correct / total_n
@torch.no_grad()
def run_one_epoch_eval(
    model: nn.Module,
    loader: DataLoader,
    kernel_l1: float,
    bias_l1: float,
    amp: bool = True,
) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_n = 0
    
    for xb, yb_int in loader:
        xb = xb.to(device, non_blocking=True)
        yb_int = yb_int.to(device, non_blocking=True)
        yb_oh = one_hot(yb_int, num_classes=10)
        
        with torch.cuda.amp.autocast(enabled=(amp and device.type == 'cuda')):
            probs = model(xb)
            loss = categorical_crossentropy_from_probs(probs, yb_oh)
            loss = loss + l1_regularization(model, kernel_l1=kernel_l1, bias_l1=bias_l1)
            
        bs = yb_int.size(0)
        total_loss += loss.detach().item() * bs
        total_correct += (probs.detach().argmax(dim=-1) == yb_int).sum().item()
        total_n += bs
        
    return total_loss / total_n, total_correct / total_n

print('Train/eval loops ready')


In [ ]:
# ==============================================================================
# 7) History + plots + confusion matrix
# ==============================================================================

def plot_history(history: Dict[str, List[float]]) -> None:
    epochs = history['epoch']
    
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['Accuracy'], label='Accuracy')
    plt.plot(epochs, history['val_Accuracy'], label='val_Accuracy')
    plt.title('Accuracy vs val_Accuracy')
    plt.xlabel('epoch')
    plt.ylabel('accuracy')
    plt.grid(True)
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['loss'], label='loss')
    plt.plot(epochs, history['val_loss'], label='val_loss')
    plt.title('Loss vs val_loss')
    plt.xlabel('epoch')
    plt.ylabel('loss')
    plt.grid(True)
    plt.legend()
    
    plt.tight_layout()
    plt.show()

@torch.no_grad()
def predict_labels(model: nn.Module, loader: DataLoader, amp: bool = True) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    y_true = []
    y_pred = []
    for xb, yb_int in loader:
        xb = xb.to(device, non_blocking=True)
        yb_int = yb_int.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(amp and device.type == 'cuda')):
            probs = model(xb)
        y_true.append(yb_int.detach().cpu().numpy())
        y_pred.append(probs.detach().argmax(dim=-1).cpu().numpy())
        y_pred.append(probs.detach().argmax(dim=-1).cpu().numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)

def plot_confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, title: str = 'Confusion Matrix') -> None:
    cm = np.zeros((10, 10), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1
        
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.show()

print('Plotting utilities ready')


In [ ]:
# ==============================================================================
# # 8) Single orchestrator function (major steps)
# ==============================================================================

def train_mnist_mlp_keras_like(
    # Data
    data_dir: str = './data',
    batch_size: int = 64,
    epochs: int = 50,
    val_split: float = 0.1,
    num_workers: int = 2,
    # Model
    kernel_init: str = 'glorot_uniform',
    bias_init: str = 'zeros',
    activation: str = 'relu',
    leaky_relu_slope: float = 0.01,
    dropout_p: float = 0.3,
    use_batchnorm: bool = True,
    # Regularization
    kernel_l1: float = 0.0,
    bias_l1: float = 0.0,
    # Advanced features
    lora_rank: Optional[int] = None,
    lora_alpha: float = 1.0,
    quantization: Optional[str] = None, # kept for API parity; must be None
    # Optimizer
    lr: float = 1e-3,
    # Early stopping
    early_stop_monitor: str = 'accuracy',
    early_stop_acc: float = 0.95,
    early_stop_loss: float = 0.1,
    early_stop_patience: int = 3,
    early_stop_min_delta: float = 0.0,
    # Performance
    amp: bool = True,
    verbose: int = 1,
    seed: int = 42,
) -> Dict[str, object]:
    """
    Single entrypoint that performs all major steps and returns artifacts.    
    Returns dict with:
    # - model
    # - history (dict of lists)
    # - total_training_time
    # - loaders
    # - y_true_val, y_pred_val
    """        
    # Step A) Validate arguments
    # ------------------------
    if quantization is not None:
        raise ValueError("quantization must be None (skipped by design in this notebook)")
    if batch_size <= 0:
        raise ValueError('batch_size must be > 0')
    if epochs <= 0:
        raise ValueError('epochs must be > 0')
        
    seed_everything(seed)
    
    # Derive validation batch size (Keras-like: can be larger for faster validation)
    # Rule: at least 128, and scale with batch_size.
    validation_batch_size = max(128, batch_size * 2)
    
    print('-----------------------------------------')
    print('MNIST MLP (Keras-like) Training')
    print('-----------------------------------------')
    print(f"device={device}")
    print(f"batch_size={batch_size} validation_batch_size={validation_batch_size} epochs={epochs}")
    print(f"kernel_init={kernel_init} bias_init={bias_init} activation={activation} dropout_p={dropout_p} batchnorm={use_batchnorm}")
    print(f"kernel_l1={kernel_l1} bias_l1={bias_l1}")
    print(f"lora_rank={lora_rank} lora_alpha={lora_alpha} quantization=None")
    print(f"optimizer=Adam lr={lr}")
    print("loss=CategoricalCrossentropy (one-hot) output=Softmax")
    print(f"early_stop_monitor={early_stop_monitor} acc>={early_stop_acc} loss<={early_stop_loss} patience={early_stop_patience}")
    
    # Step B) Data loaders
    # ------------------------
    train_loader, val_loader_tmp, test_loader_tmp = build_mnist_dataloaders(
        data_dir=data_dir,
        batch_size=batch_size,
            val_split=val_split,
            num_workers=num_workers,
            pin_memory=True,
            persistent_workers=True,
            prefetch_factor=4,
        )
        
    # Rebuild val/test loaders with derived validation_batch_size
    # (We reuse the underlying datasets from the existing loaders)
    val_ds = val_loader_tmp.dataset
    test_ds = test_loader_tmp.dataset
    
    common = dict(
        num_workers=num_workers,
        pin_memory=(device.type == 'cuda'),
        persistent_workers=(num_workers > 0),
        prefetch_factor=4 if num_workers > 0 else None,
    )
    if common['prefetch_factor'] is None:
        common.pop('prefetch_factor')
        
    val_loader = DataLoader(val_ds, batch_size=validation_batch_size, shuffle=False, drop_last=False, **common)
    test_loader = DataLoader(test_ds, batch_size=validation_batch_size, shuffle=False, drop_last=False, **common)
    
    print(f"train_batches={len(train_loader)} val_batches={len(val_loader)}")
    
    # Step C) Build model
    # ------------------------
    model = build_model(
        kernel_init=kernel_init,
        bias_init=bias_init,
        activation=activation,
        dropout_p=dropout_p,
        use_batchnorm=use_batchnorm,
        lora_rank=lora_rank,
        lora_alpha=lora_alpha,
        output_softmax=True,
    ).to(device)
    
    # Optional: channels_last doesn't matter for MLP, but kept as a general GPU hint
    # model = model.to(memory_format=torch.channels_last)
    
    print('model params:', sum(p.numel() for p in model.parameters()))
    print_cuda_memory('After model to device:')
    
    # Step D) Optimizer + AMP scaler
    # ------------------------
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scaler = torch.cuda.amp.GradScaler(enabled=(amp and device.type == 'cuda'))
    
    # Step E) Early stopping
    # ------------------------
    es = EarlyStopper(EarlyStopConfig(
        monitor=early_stop_monitor,
        acc_threshold=early_stop_acc,
        loss_threshold=early_stop_loss,
        patience=early_stop_patience,
        min_delta=early_stop_min_delta,
    ))
    
    # Step F) Training loop + history
    # ------------------------
    history = {
        'Accuracy': [],
        'loss': [],
        'val_Accuracy': [],
        'val_loss': [],
        'epoch_time': [],
        'epoch': [],
    }
    
    t0 = time.perf_counter()
    for epoch in range(1, epochs + 1):
        e0 = time.perf_counter()
        
        train_loss, train_acc = run_one_epoch_train(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            kernel_l1=kernel_l1,
            bias_l1=bias_l1,
            amp=amp,
        )
        
        val_loss, val_acc = run_one_epoch_eval(
            model=model,
            loader=val_loader,
            kernel_l1=kernel_l1,
            bias_l1=bias_l1,
            amp=amp,
        )
        
        e1 = time.perf_counter()
        epoch_time = e1 - e0
        
        history['Accuracy'].append(float(train_acc))
        history['loss'].append(float(train_loss))
        history['val_Accuracy'].append(float(val_acc))
        history['val_loss'].append(float(val_loss))
        history['epoch_time'].append(float(epoch_time))
        history['epoch'].append(int(epoch))
        
        if verbose:
            print(
                f"Epoch {epoch:03d}/{epochs} | "
                f"Accuracy={train_acc:.4f} loss={train_loss:.4f} | "
                f"val_Accuracy={val_acc:.4f} val_loss={val_loss:.4f} | "
                f"epoch_time={epoch_time:.2f}s"
            )
            
        stop, reason = es.should_stop({'val_Accuracy': val_acc, 'val_loss': val_loss})
        if stop:
            print(reason)
            break
            
    total_training_time = time.perf_counter() - t0
    print(f"Total training time: {total_training_time:.2f} seconds")
    # Step G) Plots
    # ------------------------
    plot_history(history)
    
    # Step H) Confusion matrix (validation)
    # ------------------------
    y_true_val, y_pred_val = predict_labels(model, val_loader, amp=amp)
    plot_confusion_matrix(y_true_val, y_pred_val, title='Confusion Matrix (Validation)')
    
    return {
        'model': model,
        'history': history,
        'total_training_time': total_training_time,
        'loaders': (train_loader, val_loader, test_loader),
        'y_true_val': y_true_val,
        'y_pred_val': y_pred_val,
    }

print('Orchestrator ready')


In [ ]:
# ==============================================================================
# # 9) Run training (default-like)
# ==============================================================================

# Keras-like defaults requested:
# - batch_size=64
# - epochs=50
# - dropout_p=0.3
# - optimizer=Adam
# - loss=CategoricalCrossentropy
# - early stop by accuracy >= 0.95 (default)

result = train_mnist_mlp_keras_like(
    batch_size=64,
    epochs=50,
    dropout_p=0.3,
    kernel_init='glorot_uniform',
    bias_init='zeros',
    activation='relu',
    kernel_l1=0.0,
    bias_l1=0.0,
    lora_rank=None,        # set e.g. 8 to enable LoRA
    lora_alpha=16.0,
    early_stop_monitor='accuracy',
    early_stop_acc=0.95,
    early_stop_loss=0.1,
    early_stop_patience=3,
    amp=True,
    verbose=1,
)

history = result['history']
print('History columns:', list(history.keys()))
print('Last epoch:', history['epoch'][-1])
print('Last metrics:', {k: history[k][-1] for k in ['Accuracy', 'loss', 'val_Accuracy', 'val_loss', 'epoch_time']})


## Notes on output activation (Softmax vs logits)

In Keras, `CategoricalCrossentropy` typically expects probabilities if `from_logits=False` (default), which corresponds to using a Softmax output layer.

This notebook follows that behavior:
* **Model outputs Softmax probabilities**
* **Loss is computed as categorical cross-entropy on probabilities**

PyTorch best practice often uses **logits + `CrossEntropyLoss`** (which internally applies `log_softmax`). You explicitly requested Keras-like one-hot + categorical cross-entropy, hence the custom Softmax + one-hot.
